# 💊 MediMate

**Aplikasi Pengelolaan Obat dan Pemantauan Jadwal Konsumsi Berbasis Python**

MediMate merupakan aplikasi berbasis Python yang dirancang untuk membantu pengguna mengelola informasi obat, mengatur jadwal konsumsi, memantau persediaan, serta mencatat riwayat penggunaan obat secara terstruktur.

Aplikasi ini menggunakan **Gradio** sebagai antarmuka pengguna dan **SQLite** sebagai sistem penyimpanan data. Google Colab digunakan sebagai lingkungan untuk menjalankan aplikasi melalui Jupyter Notebook (`.ipynb`).

### Petunjuk Penggunaan

1. Buka notebook menggunakan **Google Colab**.
2. Pastikan alamat repositori GitHub pada sel kode pertama telah dikonfigurasi dengan benar.
3. Pilih menu **Runtime → Run all** untuk menjalankan seluruh proses secara otomatis.
4. Sistem akan mengambil kode sumber terbaru dari GitHub, menginstal pustaka yang diperlukan, dan menjalankan aplikasi MediMate.
5. Jika aplikasi tidak terbuka secara otomatis, klik tautan **Open MediMate** yang ditampilkan setelah proses selesai.

### Mekanisme Pembaruan

Setiap kali notebook dijalankan ulang, sistem akan mengambil kode sumber terbaru dari cabang utama repositori GitHub. Dengan demikian, perubahan pada berkas `app.py`, `database.py`, `medimate_core.py`, dan berkas pendukung lainnya yang telah disimpan di GitHub akan diterapkan pada proses berikutnya.


In [ ]:
# 1/3 — Always get the newest MediMate code from GitHub
# Set this URL ONCE after creating your GitHub repository.
GITHUB_REPO = "https://github.com/Imannov-Kaba/MediMate"

from pathlib import Path
from urllib.parse import urlparse
import os, shutil, subprocess, sys

parsed = urlparse(GITHUB_REPO.rstrip('/'))
parts = [part for part in parsed.path.strip('/').split('/') if part]
if (parsed.scheme != 'https' or parsed.hostname != 'github.com' or len(parts) != 2
    or 'YOUR_GITHUB_USERNAME' in GITHUB_REPO or not parts[0] or not parts[1]):
    raise ValueError('Set GITHUB_REPO to your own GitHub URL, e.g. https://github.com/yourname/MediMate')

try:
    if '_MM_RUNNING_APP' in globals():
        _MM_RUNNING_APP.close()
except Exception:
    pass

BASE = Path('/content') if Path('/content').is_dir() else Path.cwd()
PROJECT = BASE / 'MediMate-source'
DATA_FOLDER = BASE / 'MediMate-records'
DATA_FOLDER.mkdir(parents=True, exist_ok=True)

# Re-clone every time. Never reuse the old source checkout.
if PROJECT.exists():
    shutil.rmtree(PROJECT)
subprocess.run(['git', 'clone', '--depth', '1', GITHUB_REPO, str(PROJECT)], check=True)
required = ('app.py', 'database.py', 'medimate_core.py', 'requirements.txt')
missing = [file for file in required if not (PROJECT / file).is_file()]
if missing:
    raise FileNotFoundError('Missing files in GitHub repository: ' + ', '.join(missing))

os.environ['MEDIMATE_DATA_DIR'] = str(DATA_FOLDER)
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
for module in ('app', 'database', 'medimate_core'):
    sys.modules.pop(module, None)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], cwd=PROJECT, text=True).strip()
print(f'MediMate source updated from GitHub (commit {commit}).')

In [ ]:
# 2/3 — Install dependencies and test the latest source
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(PROJECT / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-q'], check=True, cwd=PROJECT)
print('MediMate is ready.')

In [ ]:
# 3/3 — Launch MediMate in a separate browser tab
import json
from html import escape
from IPython.display import HTML, display
import gradio as gr
import app

try:
    import google.colab
    INSIDE_COLAB = True
except ImportError:
    INSIDE_COLAB = False

if INSIDE_COLAB:
    _MM_RUNNING_APP, local_url, public_url = app.medimate_app.launch(
        share=True, inline=False, prevent_thread_lock=True,
        show_error=True, footer_links=[], theme=app.THEME, css=app.CSS, auth=None
    )
    address = public_url or local_url
else:
    _MM_RUNNING_APP, local_url, public_url = app.medimate_app.launch(
        share=False, inline=False, inbrowser=True, server_name='127.0.0.1',
        prevent_thread_lock=True, show_error=True, footer_links=[],
        theme=app.THEME, css=app.CSS, auth=None
    )
    address = local_url

display(HTML(
    '<div style="max-width:650px;margin:14px auto;padding:22px;border-radius:20px;'
    'background:linear-gradient(115deg,#FFE4EF,#EEE4FF);border:1px solid #E8C8DC;'
    'font-family:Arial,sans-serif;text-align:center">'
    '<div style="font-size:20px;font-weight:800;color:#612A55">💊 MediMate is ready</div>'
    '<p style="font-size:13px;color:#573C59">Open the application in its own browser tab.</p>'
    '<a href="'+escape(address,quote=True)+'" target="_blank" rel="noopener noreferrer"'
    'style="display:inline-block;padding:14px 25px;border-radius:14px;'
    'background:#DD1777;color:#FFFFFF;text-decoration:none;font-weight:800">'
    'Open MediMate ↗</a></div>'
))
if INSIDE_COLAB:
    try:
        from google.colab import output
        output.eval_js('window.open(' + json.dumps(address) + ', "_blank", "noopener,noreferrer")')
    except Exception:
        pass

## Kode Sumber dan Penyimpanan Data

- **Pembaruan kode sumber:** Setiap kali fitur **Run all** dijalankan, sistem secara otomatis mengambil kode sumber terbaru yang telah disimpan pada repositori GitHub. Perubahan pada berkas `app.py` atau berkas pendukung lainnya dapat dilakukan melalui GitHub dan diterapkan setelah notebook dijalankan ulang.

- **Penyimpanan data:** MediMate menggunakan database **SQLite** yang disimpan pada direktori `/content/MediMate-records/` selama sesi Google Colab berlangsung. Direktori penyimpanan ini terpisah dari kode sumber aplikasi sehingga data tetap tersedia ketika kode diperbarui dalam sesi runtime yang sama. Untuk mempertahankan data setelah sesi runtime berakhir, pengguna dapat menggunakan fitur **Analytics & Backup** untuk mengekspor dan memulihkan data.

- **Identifikasi pengguna:** MediMate menggunakan sistem identifikasi berdasarkan **nama lengkap** tanpa kata sandi. Nama lengkap digunakan untuk mengakses catatan yang terkait dengan identitas tersebut. Sistem ini tidak menyediakan autentikasi akun pribadi sehingga pengguna dengan nama yang sama dapat mengakses catatan yang sama.
